# Annotate subsets

In [ ]:
import scanpy as sc
import scvi
import palantir as pt
import cellrank as cr

import pickle

from scipy import sparse
import pandas as pd
import numpy as np

import os

# Settings

In [ ]:
# Warnings Python 
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Scanpy figures
sc.set_figure_params(figsize=(5, 5), dpi_save=1200, fontsize=12, frameon=False, color_map='viridis', facecolor="white")

In [ ]:
# Root directory
os.chdir('/research/peer/fdeckert/FD20220623HSCT')

In [ ]:
# SCVI threads
scvi.settings.num_threads=32

# Import AnnData

In [ ]:
adata = sc.read_h5ad('data/object/reynolds_et_al_2021/pp.h5ad')

# Subset 

In [ ]:
sc.pl.umap(adata, color=['leiden', 'donor_id', 'sample_id', 'Status', 'Site', 'Tissue', 'Location', 'donor_id', 'Sex', 'full_clustering'], frameon=False, ncols=5, wspace=0.5, size=20)

# Lymphocyte

In [ ]:
cache = True

In [ ]:
adata_0 = adata[adata.obs['full_clustering'].isin(['ILC1_3', 'ILC1_NK', 'ILC2', 'NK', 'Tc', 'Th', 'Treg'])].copy()
adata_0 = adata_0[:, (adata_0.X>=3).sum(axis=0)>=3].copy()

In [ ]:
sc.pl.umap(adata_0, color=['leiden', 'donor_id', 'sample_id', 'Status', 'Site', 'Tissue', 'Location', 'donor_id', 'Sex', 'full_clustering'], frameon=False, ncols=5, wspace=0.5, size=20)

## SCVI

In [ ]:
if(cache): 
    
    adata_0 = sc.read_h5ad('data/object/reynolds_et_al_2021/subsets/lymphocytes/andata.h5ad')
    
else: 
    
    scvi.model.SCVI.setup_anndata(
        
        adata_0,
        batch_key='sample_id', 
        categorical_covariate_keys=None, 
        continuous_covariate_keys=['msS_RNA', 'msG2M_RNA', 'pMT_RNA']
        
    )

    model_1 = scvi.model.SCVI(

        adata_0, 
        n_latent=10, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'

    )

    model_1.train()

    model_1.save('data/object/reynolds_et_al_2021/subsets/lymphocytes', overwrite=True)

    adata_0.obsm['latent'] = model_1.get_latent_representation()

    # Dim reduction and clustering on the scanorama corrected embedding 
    sc.pp.neighbors(adata_0, n_neighbors=30, use_rep='latent', metric='euclidean')
    sc.tl.leiden(adata_0, resolution=2)
    sc.tl.umap(adata_0, min_dist=0.5)

    adata_0.write('data/object/reynolds_et_al_2021/subsets/lymphocytes/andata.h5ad')

In [ ]:
sc.pl.umap(adata_0, color=['leiden', 'donor_id', 'sample_id', 'Status', 'Site', 'Tissue', 'Location', 'donor_id', 'Sex', 'full_clustering'], frameon=False, ncols=5, wspace=0.5, size=20)

## Restore raw data 

In [ ]:
adata_0 = sc.read_h5ad('data/object/reynolds_et_al_2021/subsets/lymphocytes/andata.h5ad')

In [ ]:
adata_raw = sc.read_h5ad('../reference/scRNAseq/data/reynolds_et_al/data/human_adult_raw.h5ad')
adata_raw = adata_raw[adata_raw.obs_names.isin(adata_0.obs_names)].copy()

In [ ]:
adata_0.raw = adata_raw
adata_0 = adata_0.raw.to_adata()

## Celltypist labels 

In [ ]:
adata_0.obs['celltypist_predicted_labels_low'] = adata_0.obs['celltypist_predicted_labels_low'].apply(lambda x: x if (adata_0.obs['celltypist_predicted_labels_low'].value_counts(normalize=True)[x] * 100) > 1 else 'freq < 1 %')
adata_0.obs['celltypist_predicted_labels_high'] = adata_0.obs['celltypist_predicted_labels_high'].apply(lambda x: x if (adata_0.obs['celltypist_predicted_labels_high'].value_counts(normalize=True)[x] * 100) > 1 else 'freq < 1 %')

## Gene modules 

In [ ]:
cell_type_genes_dict = {

    r'$TRC\alpha$': [i for i in adata_0.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata_0.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata_0.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata_0.var_names if 'TRDC' in i],
    r'CD3': ['CD3E', 'CD3D', 'CD3G', 'CD247'],
    r'CD4': ['CD4'], 
    r'CD8': ['CD8A', 'CD8B'],
    r'Regulatory': ['CTLA4', 'IL2RA', 'FOXP3', 'TIGIT'], 
    r'Cytotoxic': ['KLRB1', 'GZMB', 'CCL5', 'NKG7', 'PRF1', 'GZMM'], # CD161:KLRB1
    r'TRM17': ['CCR6', 'IL7R', 'IL17A', 'MAF', 'RUNX3', 'ZNF683'], 
    r'NK': ['NCAM1', 'KLRD1', 'FCGR3A'], # CD56:NCAM1, CD94:KLRD1, CD16:FCGR3A | Possible TRGC/TRDC | CD3E has been found | NCAM1 not found | 
    r'$T_{inv}$': ['ZBTB16'], # PZLF:ZBTB16, CD4+ or DN, four classes with some having restriceted TRAC/TRBC | Marker overlap with NK | Type I T_{inv} Vα24Jα18
    r'MAIT': ['SLC4A10', 'CEBPD'], # Vα7.2Jα33, TRAV1-2 | SLC4A10 not found
    r'Residency': ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], # CD69 distinguishes memory T cell in tissue from blood
    r'Naive': ['TCF7', 'SELL', 'CCR7'], # CD127:IL7R, CD62L:SELL
    r'Experienced': ['CD44'], #  
    r'Activated': ['CD38', 'FGFBP2', 'CD5'],  
    r'Exhaustion': ['PDCD1', 'TIGIT', 'HAVCR2'], # PDCD1:PD-1, HAVCR2:TIM3
    r'Cycling': ['PCNA', 'MKI67', 'TOP2A', 'CDK1']
    
}

In [ ]:
tcr_genes_dict = {
    
    r'$TRC\alpha$': [i for i in adata_0.var_names if 'TRAC' in i],
    r'$TRC\beta$': [i for i in adata_0.var_names if 'TRBC' in i],
    r'$TRC\gamma$': [i for i in adata_0.var_names if 'TRGC' in i],
    r'$TRC\delta$': [i for i in adata_0.var_names if 'TRDC' in i],
    r'$TRV\alpha$': [i for i in adata_0.var_names if 'TRAV' in i],
    r'$TRV\beta$': [i for i in adata_0.var_names if 'TRBV' in i],
    r'$TRV\gamma$': [i for i in adata_0.var_names if 'TRGV' in i],
    r'$TRV\delta$': [i for i in adata_0.var_names if 'TRDV' in i]

}

## Sub-cluster

In [ ]:
sc.tl.leiden(adata_0, resolution=0.1, restrict_to=('leiden', ['25']), key_added='leiden_restrict')
sc.tl.leiden(adata_0, resolution=0.2, restrict_to=('leiden_restrict', ['17']), key_added='leiden_restrict')

adata_0.obs['leiden'] = adata_0.obs['leiden_restrict']
del adata_0.obs['leiden_restrict']

## Leiden annotation 

In [ ]:
leiden_cell_type_fine = {
    
    '0': r'$T_{reg}\ (0)$',
    '1': r'$NK\ CD16-\ (1)$',
    '2': r'$T_{h, memory/effector}\ (2)$',
    '3': r'$T_{h, memory/effector}\ (3)$',
    '4': r'$T_{h, memory/effector}\ (4)$',
    '5': r'$T_{inv}\ (5)$',
    '6': r'$T_{h, memory/effector}\ (6)$',
    '7': r'$T_{h, memory/effector}\ (7)$',
    '8': r'$T_{h, memory/effector}\ (8)$',
    '9': r'$T_{c, memory/effector}\ (9)$',
    '10': r'$ILC1/3 (10)$',
    '11': r'$ILC1/3 (11)$',
    '12': r'$T_{c, memory/effector}\ (12)$',
    '13': r'$T_{h, memory/effector}\ (13)$',
    '14': r'$T_{reg}\ (14)$',
    '15': r'$T_{h, memory/effector}\ (15)$',
    '16': r'$T_{h, memory/effector}\ (16)$',
    '17,0': r'$T_{h, memory/effector}\ (17,0)$',
    '17,1': r'$T_{c, memory/effector}\ (17,1)$',
    '18': r'$T_{c, memory/effector}\ (18)$',
    '19': r'$T_{h, memory/effector}\ (19)$',
    '20': r'$T_{inv}\ (20)$',
    '21': r'$T_{h, memory/effector}\ (21)$',
    '22': r'$T_{h, memory/effector}\ (22)$',
    '23': r'$T_{h, memory/effector}\ (23)$',
    '24': r'$T_{inv}\ (24)$',
    '25,0': r'$ILC2\ (25,0)$',
    '25,1': r'$NK\ CD16-\ (25,1)$',
    '26': r'$T_{h, memory/effector}\ (26)$',
    '27': r'$NK\ CD16+\ (27)$',
    '28': r'$T_{c, memory/effector}\ (28)$'

}

In [ ]:
# Mapper 
leiden_cell_type_fine = pd.DataFrame({'leiden_cell_type_fine': adata_0.obs['leiden'].map(lambda x: leiden_cell_type_fine.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(leiden_cell_type_fine, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['leiden_cell_type_fine'] = adata_0.obs['leiden_cell_type_fine'].cat.reorder_categories([
    
    r'$T_{h, memory/effector}\ (13)$',
    r'$T_{h, memory/effector}\ (15)$',
    r'$T_{h, memory/effector}\ (16)$',
    r'$T_{h, memory/effector}\ (3)$',
    r'$T_{h, memory/effector}\ (4)$',
    r'$T_{h, memory/effector}\ (6)$',
    r'$T_{h, memory/effector}\ (8)$',
    r'$T_{h, memory/effector}\ (17,0)$', 
    r'$T_{h, memory/effector}\ (19)$',
    r'$T_{h, memory/effector}\ (21)$',
    r'$T_{h, memory/effector}\ (22)$',
    r'$T_{h, memory/effector}\ (2)$',
    r'$T_{h, memory/effector}\ (7)$',
    r'$T_{h, memory/effector}\ (23)$',
    r'$T_{h, memory/effector}\ (26)$',
    
    r'$T_{reg}\ (0)$',
    r'$T_{reg}\ (14)$',
    
    r'$T_{c, memory/effector}\ (9)$',
    r'$T_{c, memory/effector}\ (12)$',
    r'$T_{c, memory/effector}\ (17,1)$',
    r'$T_{c, memory/effector}\ (18)$',
    r'$T_{c, memory/effector}\ (28)$', 
    
    r'$T_{inv}\ (24)$',
    r'$T_{inv}\ (5)$',
    r'$T_{inv}\ (20)$',
    
    r'$NK\ CD16+\ (27)$', 
    r'$NK\ CD16-\ (1)$',
    r'$NK\ CD16-\ (25,1)$', 
    r'$ILC1/3 (10)$',
    r'$ILC1/3 (11)$',
    r'$ILC2\ (25,0)$'
    
])

In [ ]:
dp = sc.pl.dotplot(adata_0, cell_type_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
dp = sc.pl.dotplot(adata_0, tcr_genes_dict, groupby='leiden_cell_type_fine', standard_scale='var', use_raw=False, dendrogram=False, return_fig=True)
dp.add_totals().style(dot_edge_color='black', dot_edge_lw=0.5, cmap='Reds').show()

In [ ]:
leiden_cell_type_main = {
    
    '0': r'$T_{reg}$',
    '1': r'$NK\ CD16-$',
    '2': r'$T_{h, memory/effector}$',
    '3': r'$T_{h, memory/effector}$',
    '4': r'$T_{h, memory/effector}$',
    '5': r'$T_{inv}$',
    '6': r'$T_{h, memory/effector}$',
    '7': r'$T_{h, memory/effector}$',
    '8': r'$T_{h, memory/effector}$',
    '9': r'$T_{c, memory/effector}$',
    '10': r'$ILC1/3$',
    '11': r'$ILC1/3$',
    '12': r'$T_{c, memory/effector}$',
    '13': r'$T_{h, memory/effector}$',
    '14': r'$T_{reg}$',
    '15': r'$T_{h, memory/effector}$',
    '16': r'$T_{h, memory/effector}$',
    '17,0': r'$T_{h, memory/effector}$',
    '17,1': r'$T_{c, memory/effector}$',
    '18': r'$T_{c, memory/effector}$',
    '19': r'$T_{h, memory/effector}$',
    '20': r'$T_{inv}$',
    '21': r'$T_{h, memory/effector}$',
    '22': r'$T_{h, memory/effector}$',
    '23': r'$T_{h, memory/effector}$',
    '24': r'$T_{inv}$',
    '25,0': r'$ILC2$',
    '25,1': r'$NK\ CD16-$',
    '26': r'$T_{h, memory/effector}$',
    '27': r'$NK\ CD16+$',
    '28': r'$T_{c, memory/effector}$'

}

In [ ]:
# Mapper 
leiden_cell_type_main = pd.DataFrame({'leiden_cell_type_main': adata_0.obs['leiden'].map(lambda x: leiden_cell_type_main.get(x, x)).astype('category')})
adata_0.obs = adata_0.obs.merge(leiden_cell_type_main, left_index=True, right_index=True, how='left')

In [ ]:
adata_0.obs['leiden_cell_type_main'] = adata_0.obs['leiden_cell_type_main'].cat.reorder_categories([
    
    r'$T_{h, memory/effector}$',
    
    r'$T_{reg}$',
    
    r'$T_{c, memory/effector}$',
    
    r'$T_{inv}$', 
    
    r'$NK\ CD16+$', 
    r'$NK\ CD16-$',
    r'$ILC1/3$', 
    r'$ILC2$'
    
])

# PAGA and pseudotime analysis 

## Helper function 

In [ ]:
def scvi_integrate(adata, model_dir=None, file=None): 
    
    adata.raw = adata
    adata = adata[:, (adata.X>=3).sum(axis=0)>=3].copy()
    
    scvi.model.SCVI.setup_anndata(
        
        adata,
        batch_key='sample_id'
        
    )
    
    model = scvi.model.SCVI(
        
        adata, 
        n_latent=10, 
        n_hidden=128, 
        n_layers=2, 
        gene_likelihood='nb'
        
    )
    
    model.train()
    
    # Write output 
    if model_dir is not None: 
        model.save(model_dir, overwrite=True)
    
    adata.obsm['X_scvi'] = model.get_latent_representation()
    
    # Write output 
    if file is not None: 
        adata.write_h5ad(file)
    
    return adata

In [ ]:
def paga_dim(adata, n_neighbors=15, use_rep='X_scvi', metric='euclidean', resolution=2, groups='leiden', layout='fr'):
    
    # SCVI based neighbors and Leiden cluster
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, metric=metric)
    sc.tl.leiden(adata, resolution=resolution)
        
    # PAGA 
    sc.tl.paga(adata, groups=groups, model='v1.2')
    sc.pl.paga(adata, layout=layout, threshold=0, plot=False)
    
    # PAGA UMAP
    sc.tl.umap(adata, init_pos='paga', min_dist=0.5)
    
    return adata

In [ ]:
def palantir_workflow(adata, celltype, celltype_column, n_components=15, knn=30): 
    
    # Save original count data 
    X = adata.X
    
    # Normalization 
    sc.pp.normalize_per_cell(adata)
    pt.preprocess.log_transform(adata)
    adata.layers['data'] = adata.X
    
    # Compute diffusion components
    dm_res = pt.utils.run_diffusion_maps(adata, n_components=n_components, knn=knn, pca_key='X_scvi')
    
    # Data imputation with MAGIC
    adata.X=sparse.csr_matrix(adata.X)
    adata.layers['data_magic'] = sparse.csr_matrix(pt.utils.run_magic_imputation(adata, dm_res))
    
    # Low dimensional embedding of the data based on the eigen gap
    ms_data = pt.utils.determine_multiscale_space(adata)
    adata.obsm['X_diffmap'] = ms_data.to_numpy()
    
    # Compute start cell
    early_cell_id = pt.utils.early_cell(adata, celltype, celltype_column=celltype_column)
    
    # Running Palantir 
    pr_res = pt.core.run_palantir(adata, early_cell=early_cell_id, knn=knn)
    
    # Set count data to X
    adata.X = X
    
    return adata

In [ ]:
def pl_summary(adata, layer='data'): 
    
    # Set layer for plotting 
    # adata.X = adata.layers[layer]
    
    # Plot 
    sc.pl.paga(adata, layout='fr', threshold=0.25, fontsize=12, node_size_scale=1.0, node_size_power=1.0, max_edge_width=0.5, color=['full_clustering', 'donor_id', 'ms_residency', 'cc_phase_class'])
    sc.pl.paga(adata, layout='fr', threshold=0.25, fontsize=12, node_size_scale=1.0, node_size_power=1.0, max_edge_width=0.5, color=['CCR6', 'MAF', 'IL7R', 'IL17A', 'RUNX3', 'ZNF683'], use_raw=True)
    sc.pl.umap(adata, color=['full_clustering', 'donor_id', 'ms_residency', 'cc_phase_class'], wspace=0.2, ncols=4)
    # sc.pl.embedding(adata, basis='X_diffmap', color=['full_clustering', 'Site', 'donor_id', 'Sex', 'cc_phase_class'], wspace=0.2, ncols=6)
    
    sc.pl.umap(adata, color=['leiden'], wspace=0.2, legend_loc='on data')

## CD4+ T-cells 

In [ ]:
if(True): 
    
    # Filter cells 
    adata_1 = adata_0[adata_0.obs['full_clustering'].isin(['Th'])].copy()
    
    # SCVI integration 
    adata_1 = scvi_integrate(adata_1, model_dir='data/object/reynolds_et_al_2021/pseudotime/model_1', file='data/object/reynolds_et_al_2021/pseudotime/scvi_1.h5ad')
    
else: 
        
    adata_1 = sc.read_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_1.h5ad')

## PAGA embedding and pseudotime 

In [ ]:
if(True): 
    
    # Set raw counts 
    adata_1 = adata_1.raw.to_adata()
    adata_1.raw = adata_1
    
    # PAGA graph and UMAP 
    adata_1 = paga_dim(adata_1, n_neighbors=15, resolution=2)
    
    # Module score 
    sc.pp.normalize_per_cell(adata_1)
    sc.pp.log1p(adata_1)
    sc.tl.score_genes(adata_1, ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], score_name='ms_residency')

#     # Palantir imputation and pseudotime 
#     adata_1 = palantir_workflow(adata_1, 'Th', 'full_clustering', n_components=15, knn=30)

#     # Save results 
#     adata_1.write_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_pt_1.h5ad')
    
#     # Plot results 
#     pl_summary(adata_1.copy(), layer='data_magic')
    
else: 
    
    adata_1 = sc.read_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_pt_1.h5ad')
    
    # Plot results 
    pl_summary(adata_1.copy(), layer='data_magic')

In [ ]:
# Plot results 
pl_summary(adata_1.copy(), layer='data_magic')

## CD8+ T-cells 

In [ ]:
if(True): 
    
    # Filter cells 
    adata_2 = adata_0[adata_0.obs['full_clustering'].isin(['Tc'])].copy()
    
    # SCVI integration 
    adata_2 = scvi_integrate(adata_2, model_dir='data/object/reynolds_et_al_2021/pseudotime/model_2', file='data/object/reynolds_et_al_2021/pseudotime/scvi_2.h5ad')
    
else: 
        
    adata_2 = sc.read_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_2.h5ad')

## PAGA embedding and pseudotime 

In [ ]:
if(True): 
    
    # Set raw counts 
    adata_2 = adata_2.raw.to_adata()
    adata_2.raw = adata_2
    
    # PAGA graph and UMAP 
    adata_2 = paga_dim(adata_2, n_neighbors=15, resolution=2)
    
    # Module score 
    sc.pp.normalize_per_cell(adata_2)
    sc.pp.log1p(adata_2)
    sc.tl.score_genes(adata_2, ['CREM', 'LMNA', 'EZR', 'ZFP36', 'CD69'], score_name='ms_residency')

#     # Palantir imputation and pseudotime 
#     adata_2 = palantir_workflow(adata_2, '1', 'leiden', n_components=100, knn=30)

#     # Save results 
#     adata_2.write_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_pt_2.h5ad')
    
#     # Plot results 
#     pl_summary(adata_2.copy(), layer='data_magic')
    
else: 
    
    adata_2 = sc.read_h5ad('data/object/reynolds_et_al_2021/pseudotime/scvi_pt_2.h5ad')
    
    # Plot results 
    pl_summary(adata_2.copy(), layer='data_magic')

In [ ]:
# Plot results 
pl_summary(adata_2.copy(), layer='data_magic')